In [ ]:
import os
import sys
from pathlib import Path
import nbformat
from flask import Flask, jsonify, request
from flask_cors import CORS

# Get the project root directory
# When running as notebook, Path.cwd() should be the backend directory
# So project root is parent of cwd
current_dir = Path.cwd()
if current_dir.name == "backend":
    project_root = current_dir.parent
else:
    # Fallback: assume we're in project root
    project_root = current_dir

backend_dir = project_root / "backend"
init_notebook_path = backend_dir / "lib" / "data" / "init.ipynb"

# Load and execute the init notebook
nb = nbformat.read(str(init_notebook_path), as_version=4)
namespace = {}
for cell in nb.cells:
    if cell.cell_type == "code":
        # Replace relative path with absolute path
        code = cell.source
        if "../../../data/data.h5" in code:
            data_path = project_root / "data" / "data.h5"
            # Replace the path string, handling both quoted and unquoted cases
            import re
            # Replace "path" or 'path' with the absolute path using a callable to avoid escape processing
            code = re.sub(r'["\']\.\.\/\.\.\/\.\.\/data\/data\.h5["\']', lambda m: f'"{data_path.as_posix()}"', code)
        exec(code, namespace)

# Extract functions and variables we need
create_plant = namespace.get('create_plant')
create_empID = namespace.get('create_empID')
hruuid = namespace.get('hruuid')
h5py = namespace.get('h5py')
datetime = namespace.get('datetime')

# Initialize Flask app
# Provide explicit root_path and import_name since we're executing from a notebook
import os
app = Flask('server', root_path=str(backend_dir))
CORS(app, origins=["http://localhost:4321", "http://localhost:4322", "http://localhost:3000"])

# HDF5 file path
data_file_path = project_root / "data" / "data.h5"


In [ ]:
def get_h5_file():
    """Open and return the HDF5 file handle with retry logic for Windows file locks"""
    import time
    max_retries = 5
    retry_delay = 0.5
    
    for attempt in range(max_retries):
        try:
            return h5py.File(str(data_file_path), "a")
        except OSError as e:
            if attempt < max_retries - 1:
                print(f"File lock error (attempt {attempt + 1}/{max_retries}), retrying in {retry_delay}s...")
                time.sleep(retry_delay)
            else:
                print(f"Failed to open HDF5 file after {max_retries} attempts.")
                print("Try running: backend/kill_processes.ps1")
                raise

def create_plant_with_context(f):
    """Create a plant using the provided file handle"""
    # Set up the context that create_plant expects
    namespace['file'] = f
    namespace['plant_group'] = f.require_group("plants")
    # Execute create_plant in the updated namespace
    exec('plant = create_plant()', namespace)
    return namespace['plant']

def create_sequencer_effect(f, effect_type: str, row: int, col: int, properties: dict = None):
    """Create a sequencer effect group in sequencer_effects_properties and return its UUID"""
    effects_props_group = f.require_group("sequencer_effects_properties")
    effect_uuid = hruuid.generate()
    sequencer_effect = effects_props_group.require_group(effect_uuid)
    sequencer_effect.attrs["effect_type"] = effect_type
    sequencer_effect.attrs["sequencer_row"] = row
    sequencer_effect.attrs["sequencer_col"] = col
    sequencer_effect.attrs["timestamp"] = datetime.now().isoformat()
    
    # Store properties as attributes
    if properties:
        for key, value in properties.items():
            sequencer_effect.attrs[f"prop_{key}"] = value
    
    # Return the UUID
    return effect_uuid

def get_sequencer_grid():
    """Read sequencer dataset and return as 2x12 grid"""
    with get_h5_file() as f:
        if "sequencer" not in f.keys():
            # Create sequencer if it doesn't exist
            f.create_dataset("sequencer", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer"]
        # Handle 3D shape (2,12,0) or 2D shape (2,12)
        if len(seq.shape) == 3:
            if seq.shape[2] == 0:
                # Empty 3D array, return empty 2x12 grid
                grid = [["" for _ in range(12)] for _ in range(2)]
            else:
                # Use first slice
                grid = seq[:, :, 0].tolist()
                # Convert bytes to strings if needed, normalize empty values to ""
                grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else str(cell) if cell else "" for cell in row] for row in grid]
                # Normalize: ensure empty strings, not None or whitespace
                grid = [[cell.strip() if isinstance(cell, str) and cell.strip() else "" for cell in row] for row in grid]
        else:
            # 2D array
            grid = seq[:].tolist()
            # Convert bytes to strings if needed, normalize empty values to ""
            grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else str(cell) if cell else "" for cell in row] for row in grid]
            # Normalize: ensure empty strings, not None or whitespace
            grid = [[cell.strip() if isinstance(cell, str) and cell.strip() else "" for cell in row] for row in grid]
        return grid

def set_sequencer_grid(grid):
    """Write 2x12 grid to sequencer dataset"""
    with get_h5_file() as f:
        if "sequencer" not in f.keys():
            f.create_dataset("sequencer", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer"]
        # Reshape to 2D if needed
        if len(seq.shape) == 3:
            # Resize third dimension to 1 if needed
            if seq.shape[2] == 0:
                seq.resize((2, 12, 1))
            # Ensure all cells are strings, then encode
            encoded_grid = []
            for row in grid:
                encoded_row = []
                for cell in row:
                    if cell is None:
                        encoded_row.append(b'')
                    elif isinstance(cell, bytes):
                        encoded_row.append(cell)
                    elif isinstance(cell, str):
                        encoded_row.append(cell.encode('utf-8'))
                    else:
                        encoded_row.append(str(cell).encode('utf-8'))
                encoded_grid.append(encoded_row)
            seq[:, :, 0] = encoded_grid
        else:
            # Ensure all cells are strings, then encode
            encoded_grid = []
            for row in grid:
                encoded_row = []
                for cell in row:
                    if cell is None:
                        encoded_row.append(b'')
                    elif isinstance(cell, bytes):
                        encoded_row.append(cell)
                    elif isinstance(cell, str):
                        encoded_row.append(cell.encode('utf-8'))
                    else:
                        encoded_row.append(str(cell).encode('utf-8'))
                encoded_grid.append(encoded_row)
            seq[:] = encoded_grid
        
        # Explicitly flush to ensure data is written
        f.flush()

def get_effects_grid():
    """Read sequencer_effects dataset and return as 2x12 grid"""
    with get_h5_file() as f:
        if "sequencer_effects" not in f.keys():
            # Create sequencer_effects if it doesn't exist
            f.create_dataset("sequencer_effects", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer_effects"]
        # Handle 3D shape (2,12,0) or 2D shape (2,12)
        if len(seq.shape) == 3:
            if seq.shape[2] == 0:
                # Empty 3D array, return empty 2x12 grid
                grid = [["" for _ in range(12)] for _ in range(2)]
            else:
                # Use first slice
                grid = seq[:, :, 0].tolist()
                # Convert bytes to strings if needed
                grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else (cell if cell else "") for cell in row] for row in grid]
        else:
            # 2D array
            grid = seq[:].tolist()
            # Convert bytes to strings if needed
            grid = [[cell.decode('utf-8') if isinstance(cell, bytes) else (cell if cell else "") for cell in row] for row in grid]
        return grid

def set_effects_grid(grid):
    """Write 2x12 grid to sequencer_effects dataset"""
    with get_h5_file() as f:
        if "sequencer_effects" not in f.keys():
            f.create_dataset("sequencer_effects", shape=(2, 12), maxshape=(2, 12, None), 
                            dtype=h5py.string_dtype(encoding='utf-8'))
        
        seq = f["sequencer_effects"]
        # Reshape to 2D if needed
        if len(seq.shape) == 3:
            # Resize third dimension to 1 if needed
            if seq.shape[2] == 0:
                seq.resize((2, 12, 1))
            seq[:, :, 0] = [[cell.encode('utf-8') if isinstance(cell, str) else cell for cell in row] for row in grid]
        else:
            seq[:] = [[cell.encode('utf-8') if isinstance(cell, str) else cell for cell in row] for row in grid]


In [ ]:
@app.route('/api/plants', methods=['GET'])
def get_plants():
    """Get all plants with their IDs and timestamps"""
    try:
        with get_h5_file() as f:
            plants_group = f.require_group("plants")
            plants = []
            for plant_id in plants_group.keys():
                plant = plants_group[plant_id]
                added_timestamp = plant.attrs.get("added_timestamp", "")
                if isinstance(added_timestamp, bytes):
                    added_timestamp = added_timestamp.decode('utf-8')
                plants.append({
                    "id": plant_id,
                    "added_timestamp": added_timestamp
                })
        return jsonify({"plants": plants}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/plants', methods=['POST'])
def create_plant_endpoint():
    """Create a new plant and return its ID"""
    try:
        with get_h5_file() as f:
            plant = create_plant_with_context(f)
            plant_id = plant.name.split('/')[-1]  # Get the plant ID from the group name
            return jsonify({"plant_id": plant_id}), 201
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/sequencer', methods=['GET'])
def get_sequencer():
    """Get current sequencer grid state"""
    try:
        grid = get_sequencer_grid()
        return jsonify({"sequencer": grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/sequencer', methods=['PUT'])
def update_sequencer():
    """Update sequencer grid position"""
    try:
        data = request.get_json()
        row = data.get('row')
        col = data.get('col')
        plant_id = data.get('plant_id', '')  # Empty string to clear
        
        if row is None or col is None:
            return jsonify({"error": "row and col are required"}), 400
        
        if row < 0 or row >= 2 or col < 0 or col >= 12:
            return jsonify({"error": "row must be 0-1, col must be 0-11"}), 400
        
        grid = get_sequencer_grid()
        grid[row][col] = plant_id
        set_sequencer_grid(grid)
        
        return jsonify({"sequencer": grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/sequencer/auto-populate', methods=['POST'])
def auto_populate_sequencer():
    """Auto-populate empty sequencer slots with new plants, skipping tiles with effects"""
    try:
        grid = get_sequencer_grid()
        effects_grid = get_effects_grid()
        count = 0
        
        with get_h5_file() as f:
            for row in range(2):
                for col in range(12):
                    # Skip tiles that have effects (to prevent hanoi sorting issues)
                    if effects_grid[row][col]:
                        continue
                    if not grid[row][col]:  # Empty slot without effect
                        # Create a new plant
                        plant_group = create_plant_with_context(f)
                        # Extract plant ID from the group name
                        plant_id = plant_group.name.split('/')[-1]
                        grid[row][col] = plant_id
                        count += 1
        
        set_sequencer_grid(grid)
        return jsonify({"success": True, "count": count, "sequencer": grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/em-effects/auto-set', methods=['POST'])
def auto_set_em_effects():
    """Auto-set random EM effects for empty slots in EM exposure zone (cols 8-11)"""
    try:
        import random
        grid = get_effects_grid()
        count = 0
        effect_types = ['AC', 'DC', 'AMF', 'CMF']
        
        with get_h5_file() as f:
            for row in range(2):
                for col in range(8, 12):  # Only EM exposure columns
                    if not grid[row][col]:  # Empty slot
                        effect_type = random.choice(effect_types)
                        
                        # Generate random properties based on effect type
                        properties = {}
                        if effect_type == 'AC':
                            properties = {
                                'frequency': random.randint(50, 60),
                                'voltage': random.randint(100, 240),
                                'phase': random.randint(0, 360)
                            }
                        elif effect_type == 'DC':
                            properties = {
                                'voltage': random.randint(5, 24),
                                'current': random.uniform(0.1, 2.0)
                            }
                        elif effect_type == 'AMF':
                            properties = {
                                'frequency': random.randint(1, 100),
                                'amplitude': random.uniform(0.5, 5.0),
                                'phase': random.randint(0, 360)
                            }
                        elif effect_type == 'CMF':
                            properties = {
                                'strength': random.uniform(0.1, 1.0),
                                'direction': random.choice(['N', 'S', 'E', 'W'])
                            }
                        
                        # Create sequencer effect
                        uuid = create_sequencer_effect(f, effect_type, row, col, properties)
                        grid[row][col] = effect_type
                        count += 1
        
        set_effects_grid(grid)
        return jsonify({"success": True, "count": count, "effects": grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/effects', methods=['GET'])
def get_effects():
    """Get current effects grid state"""
    try:
        grid = get_effects_grid()
        return jsonify({"effects": grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/effects', methods=['PUT'])
def update_effects():
    """Update effects grid position"""
    try:
        data = request.get_json()
        row = data.get('row')
        col = data.get('col')
        effect = data.get('effect', '')  # Empty string to clear
        properties = data.get('properties', {})
        
        if row is None or col is None:
            return jsonify({"error": "row and col are required"}), 400
        
        if row < 0 or row >= 2 or col < 0 or col >= 12:
            return jsonify({"error": "row must be 0-1, col must be 0-11"}), 400
        
        # Only allow effects in EM exposure zone (cols 8-11)
        if col < 8 and effect:
            return jsonify({"error": "Effects can only be placed in EM exposure zone (cols 8-11)"}), 400
        
        grid = get_effects_grid()
        
        # If setting an effect, create or update the effect entry
        if effect:
            with get_h5_file() as f:
                uuid = create_sequencer_effect(f, effect, row, col, properties)
        
        grid[row][col] = effect
        set_effects_grid(grid)
        
        return jsonify({"effects": grid}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/effects/<int:row>/<int:col>/properties', methods=['GET'])
def get_effect_properties(row, col):
    """Get properties for effect at specific position"""
    try:
        grid = get_effects_grid()
        effect_type = grid[row][col] if row < len(grid) and col < len(grid[row]) else ""
        
        if not effect_type:
            return jsonify({"error": "No effect at this position"}), 404
        
        with get_h5_file() as f:
            effects_props_group = f.require_group("sequencer_effects_properties")
            
            # Find the effect for this position
            for uuid in effects_props_group.keys():
                effect_group = effects_props_group[uuid]
                if effect_group.attrs.get('row') == row and effect_group.attrs.get('col') == col:
                    props = {}
                    props_group = effect_group.get('properties')
                    if props_group:
                        for key in props_group.attrs.keys():
                            val = props_group.attrs[key]
                            if isinstance(val, bytes):
                                val = val.decode('utf-8')
                            props[key] = val
                    return jsonify({
                        "effect_type": effect_type,
                        "properties": props
                    }), 200
            
            return jsonify({"error": "Effect properties not found"}), 404
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/effects/<int:row>/<int:col>/properties', methods=['PUT'])
def update_effect_properties(row, col):
    """Update properties for effect at specific position"""
    try:
        data = request.get_json()
        properties = data.get('properties', {})
        
        with get_h5_file() as f:
            effects_props_group = f.require_group("sequencer_effects_properties")
            
            # Find and update the effect for this position
            for uuid in effects_props_group.keys():
                effect_group = effects_props_group[uuid]
                if effect_group.attrs.get('row') == row and effect_group.attrs.get('col') == col:
                    props_group = effect_group.require_group('properties')
                    for key, val in properties.items():
                        props_group.attrs[key] = val
                    return jsonify({"success": True}), 200
            
            return jsonify({"error": "Effect not found at this position"}), 404
    except Exception as e:
        return jsonify({"error": str(e)}), 500

In [ ]:
# Growth calculation and test queue management functions
import random
import math

# Test queue configuration
MAX_RETESTS = 5  # Maximum times to test each configuration
test_queue = []  # List of {config: (effect_type, properties), priority: int}

def calculate_plant_growth(plant_id: str, tick: int, effect_type: str, effect_properties: dict) -> float:
    """
    Calculate simulated plant growth based on EM effect parameters.
    Returns a growth value (0-1 scale).
    
    In reality this would come from actual measurements - this is a simulation
    that creates plausible-looking data for testing the system.
    """
    import hashlib
    
    # Base growth from plant genetics (deterministic based on plant_id)
    plant_hash = int(hashlib.md5(plant_id.encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
    base_growth = 0.3 + 0.4 * plant_hash  # 0.3 to 0.7 base
    
    # Time factor (growth increases over time, with diminishing returns)
    time_factor = 1.0 + 0.1 * math.log1p(tick / 100)
    
    # Effect modifier based on effect type and properties
    effect_modifier = 1.0
    
    if effect_type == 'AC':
        freq = effect_properties.get('frequency', effect_properties.get('ac_frequency', 60))
        voltage = effect_properties.get('voltage', effect_properties.get('ac_voltage', 120))
        # Optimal frequency around 50-60 Hz, optimal voltage around 100-150V
        freq_factor = 1.0 - abs(freq - 55) / 200
        voltage_factor = 1.0 - abs(voltage - 125) / 300
        effect_modifier = 1.0 + 0.3 * freq_factor * voltage_factor
        
    elif effect_type == 'DC':
        voltage = effect_properties.get('voltage', effect_properties.get('dc_voltage', 12))
        current = effect_properties.get('current', effect_properties.get('dc_current', 1.0))
        # Optimal DC around 10-15V, 0.5-1.5A
        voltage_factor = 1.0 - abs(voltage - 12) / 50
        current_factor = 1.0 - abs(current - 1.0) / 5
        effect_modifier = 1.0 + 0.25 * voltage_factor * current_factor
        
    elif effect_type == 'AMF':
        freq = effect_properties.get('frequency', effect_properties.get('amf_frequency', 50))
        amplitude = effect_properties.get('amplitude', effect_properties.get('amf_amplitude', 2.5))
        # Optimal AMF around 25-75 Hz, 2-3 amplitude
        freq_factor = 1.0 - abs(freq - 50) / 150
        amp_factor = 1.0 - abs(amplitude - 2.5) / 5
        effect_modifier = 1.0 + 0.35 * freq_factor * amp_factor
        
    elif effect_type == 'CMF':
        strength = effect_properties.get('strength', effect_properties.get('cmf_strength', 1.0))
        # Optimal CMF strength around 0.5-1.5
        strength_factor = 1.0 - abs(strength - 1.0) / 3
        effect_modifier = 1.0 + 0.2 * strength_factor
    
    # Add some random noise (measurement uncertainty)
    noise = random.gauss(0, 0.05)
    
    # Calculate final growth
    growth = base_growth * time_factor * effect_modifier + noise
    
    # Clamp to reasonable range
    return max(0.0, min(1.0, growth))

def record_growth_data(plant_id: str, tick: int, growth: float, effect_type: str, effect_properties: dict):
    """
    Record growth data point to HDF5 storage.
    Creates a new observation entry in the plant_growth group.
    """
    import sys
    try:
        with get_h5_file() as f:
            growth_group = f.require_group("plant_growth")
            
            # Create unique observation ID
            obs_id = f"{plant_id}_{tick}_{hruuid.generate()[:8]}"
            obs = growth_group.require_group(obs_id)
            
            # Store basic data
            obs.attrs['plant_id'] = plant_id
            obs.attrs['tick'] = tick
            obs.attrs['growth'] = growth
            obs.attrs['effect_type'] = effect_type or 'Control'
            obs.attrs['timestamp'] = datetime.now().isoformat()
            
            # Store all effect properties with prefixes
            if effect_properties:
                for key, value in effect_properties.items():
                    # Normalize key names
                    if key.startswith(('ac_', 'dc_', 'amf_', 'cmf_')):
                        obs.attrs[key] = value
                    else:
                        # Add prefix based on effect type
                        prefix = effect_type.lower() + '_' if effect_type else ''
                        obs.attrs[prefix + key] = value
            
            print(f"[GROWTH] Recorded: {plant_id[:8]}... tick={tick} growth={growth:.3f} effect={effect_type}", file=sys.stderr)
            
    except Exception as e:
        print(f"[ERROR] Failed to record growth data: {e}", file=sys.stderr)

def get_config_test_count(effect_type: str, properties: dict) -> int:
    """
    Count how many times a specific configuration has been tested.
    """
    try:
        with get_h5_file() as f:
            if 'plant_growth' not in f:
                return 0
            
            growth_group = f['plant_growth']
            count = 0
            
            for obs_key in growth_group.keys():
                obs = growth_group[obs_key]
                obs_effect_type = obs.attrs.get('effect_type', '')
                if isinstance(obs_effect_type, bytes):
                    obs_effect_type = obs_effect_type.decode('utf-8')
                
                if obs_effect_type == effect_type:
                    # Check if properties match (approximately)
                    matches = True
                    for key, value in properties.items():
                        attr_key = key if key.startswith(effect_type.lower() + '_') else f"{effect_type.lower()}_{key}"
                        obs_value = obs.attrs.get(attr_key, None)
                        if obs_value is not None:
                            # Allow 10% tolerance for numeric values
                            if isinstance(value, (int, float)) and isinstance(obs_value, (int, float)):
                                if abs(value - obs_value) > abs(value * 0.1 + 0.1):
                                    matches = False
                                    break
                    if matches:
                        count += 1
            
            return count
    except Exception as e:
        return 0

def update_test_queue():
    """
    Update the test queue with configurations that need more testing.
    Prioritizes configurations with fewer data points.
    """
    global test_queue
    import sys
    
    # Generate candidate configurations
    candidates = []
    
    # AC configurations
    for freq in [50, 55, 60, 100, 200]:
        for voltage in [100, 120, 150, 200, 240]:
            config = ('AC', {'frequency': freq, 'voltage': voltage, 'phase': 0})
            test_count = get_config_test_count('AC', config[1])
            if test_count < MAX_RETESTS:
                priority = (MAX_RETESTS - test_count) * 100 + random.randint(0, 50)
                candidates.append({'config': config, 'priority': priority, 'test_count': test_count})
    
    # DC configurations
    for voltage in [5, 10, 12, 15, 24]:
        for current in [0.5, 1.0, 1.5, 2.0]:
            config = ('DC', {'voltage': voltage, 'current': current})
            test_count = get_config_test_count('DC', config[1])
            if test_count < MAX_RETESTS:
                priority = (MAX_RETESTS - test_count) * 100 + random.randint(0, 50)
                candidates.append({'config': config, 'priority': priority, 'test_count': test_count})
    
    # AMF configurations
    for freq in [25, 50, 75, 100]:
        for amplitude in [1.0, 2.0, 2.5, 3.0, 4.0]:
            config = ('AMF', {'frequency': freq, 'amplitude': amplitude, 'phase': 0})
            test_count = get_config_test_count('AMF', config[1])
            if test_count < MAX_RETESTS:
                priority = (MAX_RETESTS - test_count) * 100 + random.randint(0, 50)
                candidates.append({'config': config, 'priority': priority, 'test_count': test_count})
    
    # CMF configurations
    for strength in [0.5, 1.0, 1.5, 2.0]:
        config = ('CMF', {'strength': strength, 'direction': 0})
        test_count = get_config_test_count('CMF', config[1])
        if test_count < MAX_RETESTS:
            priority = (MAX_RETESTS - test_count) * 100 + random.randint(0, 50)
            candidates.append({'config': config, 'priority': priority, 'test_count': test_count})
    
    # Sort by priority (highest first)
    candidates.sort(key=lambda x: -x['priority'])
    
    # Keep top candidates
    test_queue = candidates[:20]  # Keep top 20 configs
    
    if len(test_queue) > 0:
        print(f"[QUEUE] Top config: {test_queue[0]['config'][0]} with {test_queue[0]['test_count']} tests", file=sys.stderr)

In [ ]:
import threading
import time
import math
import queue

# Tile allocation constants based on paper
# First 8 columns (0-7): Storage/Observation area  
# Last 4 columns (8-11): EM exposure zones (swap space available)
STORAGE_COLS = list(range(0, 8))  # Columns 0-7 for storage
EM_EXPOSURE_COLS = list(range(8, 12))  # Columns 8-11 for EM exposure

# Robot configuration: 2 robots per row (4 total) for better parallelism
# Robots 0,1 serve row 0, Robots 2,3 serve row 1
# SPEED INCREASED 1000x for faster data collection
ROBOT_SPEED = 500.0  # columns per tick (was 0.5, now 1000x faster)
OBSERVATION_TICKS = 1  # ticks to observe a plant (was 3)
PICKUP_TICKS = 1  # ticks to pick up a plant (was 2)
PUTDOWN_TICKS = 1  # ticks to put down a plant (was 2)
OBSERVATION_COOLDOWN_TICKS = 5  # ticks before a plant can be observed again (was 30)
EM_EXPOSURE_TICKS = 3  # How long plants need to be in EM zone (was 20)

# Robot target assignments to prevent collisions
# Each robot has a preferred EM column to target
ROBOT_EM_PREFERENCES = {
    0: [8, 9, 10, 11],   # Robot 0 prefers col 8
    1: [9, 10, 11, 8],   # Robot 1 prefers col 9  
    2: [10, 11, 8, 9],   # Robot 2 prefers col 10
    3: [11, 8, 9, 10]    # Robot 3 prefers col 11
}

# Simulation state (in-memory for now)
simulation_state = {
    'running': False,
    'tick': 0,
    'speed': 1.0,  # ticks per second
    'robots': [
        {'id': 0, 'row': 0, 'col': 0.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0},
        {'id': 1, 'row': 0, 'col': 4.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0},
        {'id': 2, 'row': 1, 'col': 0.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0},
        {'id': 3, 'row': 1, 'col': 4.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0}
    ],
    'observation_queue': [],  # Plants queued for observation
    'observation_station': {'row': -1, 'col': 11},  # Camera position at rightmost column (physical limitation)
    'plants_observed': [],  # Plants that have been observed this cycle
    'last_tick_time': None
}

simulation_lock = threading.Lock()
simulation_thread = None

# SSE Data Buffer
_growth_data_buffer = [] # List of new data points
_growth_data_lock = threading.Lock()
_new_data_event = threading.Event()

# Track which EM columns are currently being targeted by robots
robot_em_targets = {}  # {robot_id: target_col}

def save_simulation_state():
    """Persist simulation state to HDF5 for recovery after restarts"""
    import json
    with get_h5_file() as f:
        # Store in simulation_state group
        sim_group = f.require_group("simulation_metadata")
        
        # Save as JSON string in attributes
        sim_group.attrs['tick'] = simulation_state['tick']
        sim_group.attrs['running'] = simulation_state['running']
        sim_group.attrs['speed'] = simulation_state['speed']
        sim_group.attrs['robots'] = json.dumps(simulation_state['robots'])
        sim_group.attrs['observation_queue'] = json.dumps(simulation_state['observation_queue'])
        sim_group.attrs['plants_observed'] = json.dumps(simulation_state['plants_observed'])

def load_simulation_state():
    """Load simulation state from HDF5 if available"""
    import json
    try:
        with get_h5_file() as f:
            if "simulation_metadata" in f.keys():
                sim_group = f["simulation_metadata"]
                
                # Load saved state
                if 'tick' in sim_group.attrs:
                    simulation_state['tick'] = int(sim_group.attrs['tick'])
                if 'running' in sim_group.attrs:
                    simulation_state['running'] = bool(sim_group.attrs['running'])
                if 'speed' in sim_group.attrs:
                    simulation_state['speed'] = float(sim_group.attrs['speed'])
                if 'robots' in sim_group.attrs:
                    simulation_state['robots'] = json.loads(sim_group.attrs['robots'])
                if 'observation_queue' in sim_group.attrs:
                    simulation_state['observation_queue'] = json.loads(sim_group.attrs['observation_queue'])
                if 'plants_observed' in sim_group.attrs:
                    simulation_state['plants_observed'] = json.loads(sim_group.attrs['plants_observed'])
                
                print(f"✓ Loaded simulation state: tick={simulation_state['tick']}, running={simulation_state['running']}")
    except Exception as e:
        print(f"Could not load simulation state: {e}")
        print("Starting with fresh simulation state")

# Load saved state on startup
load_simulation_state()

def get_tile_allocation():
    """Return tile allocation info for frontend"""
    return {
        'storage_cols': STORAGE_COLS,
        'em_exposure_cols': EM_EXPOSURE_COLS,
        'observation_station': simulation_state['observation_station']
    }

def calculate_travel_time(from_col, to_col):
    """Calculate ticks needed to travel between columns"""
    distance = abs(to_col - from_col)
    return math.ceil(distance / ROBOT_SPEED)

def find_available_robot(row):
    """Find an idle robot for the given row"""
    for robot in simulation_state['robots']:
        if robot['row'] == row and robot['state'] == 'idle':
            return robot
    return None

def get_preferred_em_col(robot_id, grid, row):
    """Get the preferred EM column for a robot that is not occupied or targeted"""
    preferences = ROBOT_EM_PREFERENCES.get(robot_id, EM_EXPOSURE_COLS)
    
    # Get columns currently being targeted by other robots
    targeted_cols = set(robot_em_targets.values())
    
    for col in preferences:
        # Check if column is empty in grid and not being targeted
        if not grid[row][col] and col not in targeted_cols:
            return col
    
    # If all preferred cols are taken, return any empty one
    for col in EM_EXPOSURE_COLS:
        if not grid[row][col] and col not in targeted_cols:
            return col
    
    return None

def update_robot(robot):
    """Update a single robot's state for one tick"""
    global robot_em_targets
    
    if robot['state'] == 'idle':
        return
    
    if robot['state'] == 'moving_to_pickup':
        # Move toward target
        if robot['target_col'] is not None:
            if robot['col'] < robot['target_col']:
                robot['col'] = min(robot['col'] + ROBOT_SPEED, robot['target_col'])
            elif robot['col'] > robot['target_col']:
                robot['col'] = max(robot['col'] - ROBOT_SPEED, robot['target_col'])
            
            if robot['col'] == robot['target_col']:
                robot['state'] = 'picking_up'
                robot['ticks_remaining'] = PICKUP_TICKS
    
    elif robot['state'] == 'picking_up':
        robot['ticks_remaining'] -= 1
        if robot['ticks_remaining'] <= 0:
            # Pick up complete - get plant from grid
            grid = get_sequencer_grid()
            col = int(robot['target_col'] + 0.5)  # Consistent rounding
            pickup_success = False
            if 0 <= col < 12 and grid[robot['row']][col]:
                robot['holding_plant'] = grid[robot['row']][col]
                grid[robot['row']][col] = ''
                set_sequencer_grid(grid)
                pickup_success = True
            
            if not pickup_success:
                # Pickup failed (no plant at location) - go back to idle
                robot['state'] = 'idle'
                robot['target_col'] = None
                robot['task_type'] = None
                robot['em_target_col'] = None
                # Clear targeting
                if robot['id'] in robot_em_targets:
                    del robot_em_targets[robot['id']]
                return
            
            # Check task type: EM exposure vs observation
            if robot.get('task_type') == 'em_exposure':
                # Move to EM zone to place plant
                robot['state'] = 'moving_to_em_zone'
                robot['target_col'] = robot.get('em_target_col', EM_EXPOSURE_COLS[0])
            else:
                # Normal observation task
                robot['state'] = 'moving_to_observe'
                robot['target_col'] = simulation_state['observation_station']['col']
    
    elif robot['state'] == 'moving_to_em_zone':
        # Moving plant to EM zone for exposure
        target = robot['target_col']
        if robot['col'] < target:
            robot['col'] = min(robot['col'] + ROBOT_SPEED, target)
        elif robot['col'] > target:
            robot['col'] = max(robot['col'] - ROBOT_SPEED, target)
        
        if robot['col'] == target:
            robot['state'] = 'placing_in_em_zone'
            robot['ticks_remaining'] = PUTDOWN_TICKS
    
    elif robot['state'] == 'placing_in_em_zone':
        robot['ticks_remaining'] -= 1
        if robot['ticks_remaining'] <= 0:
            # Place plant in EM zone
            if robot['holding_plant']:
                grid = get_sequencer_grid()
                col = int(robot['target_col'] + 0.5)
                placed = False
                
                if 0 <= col < 12 and not grid[robot['row']][col]:
                    grid[robot['row']][col] = robot['holding_plant']
                    set_sequencer_grid(grid)
                    
                    # Track EM exposure start
                    from datetime import datetime
                    em_zone_exposure[robot['holding_plant']] = {
                        'tick_placed': simulation_state['tick'],
                        'row': robot['row'],
                        'col': col,
                        'original_storage_col': int(robot.get('original_col', 0)),
                        'timestamp': datetime.now().isoformat()
                    }
                    placed = True
                    robot['holding_plant'] = None
                else:
                    # EM zone slot occupied - find alternative using preferences
                    import sys
                    print(f"[WARNING] Robot {robot['id']} target col {col} occupied, finding alternative", file=sys.stderr)
                    
                    # Get targeted columns to avoid
                    targeted_cols = set(robot_em_targets.values())
                    
                    # Try other EM slots in same row, respecting preferences
                    for alt_col in ROBOT_EM_PREFERENCES.get(robot['id'], EM_EXPOSURE_COLS):
                        if alt_col != col and not grid[robot['row']][alt_col] and alt_col not in targeted_cols:
                            grid[robot['row']][alt_col] = robot['holding_plant']
                            set_sequencer_grid(grid)
                            em_zone_exposure[robot['holding_plant']] = {
                                'tick_placed': simulation_state['tick'],
                                'row': robot['row'],
                                'col': alt_col,
                                'original_storage_col': int(robot.get('original_col', 0))
                            }
                            placed = True
                            robot['holding_plant'] = None
                            break
                    
                    if not placed:
                        # No EM slots available - return to storage instead
                        storage_col = int(robot.get('original_col', 0))
                        if 0 <= storage_col < 8 and not grid[robot['row']][storage_col]:
                            grid[robot['row']][storage_col] = robot['holding_plant']
                            set_sequencer_grid(grid)
                            placed = True
                            robot['holding_plant'] = None
                        else:
                            # Try any empty storage slot
                            for alt_col in STORAGE_COLS:
                                if not grid[robot['row']][alt_col]:
                                    grid[robot['row']][alt_col] = robot['holding_plant']
                                    set_sequencer_grid(grid)
                                    placed = True
                                    robot['holding_plant'] = None
                                    break
                    
                    if not placed:
                        # CRITICAL: No space anywhere - keep holding and retry later
                        print(f"[CRITICAL] Robot {robot['id']} cannot place plant anywhere!", file=sys.stderr)
                        robot['state'] = 'idle'
                        robot['target_col'] = None
                        robot['task_type'] = None
                        robot['em_target_col'] = None
                        if robot['id'] in robot_em_targets:
                            del robot_em_targets[robot['id']]
                        return
            
            # Reset task state and go idle
            if not robot['holding_plant']:
                robot['task_type'] = None
                robot['em_target_col'] = None
                robot['state'] = 'idle'
                robot['target_col'] = None
                # Clear targeting
                if robot['id'] in robot_em_targets:
                    del robot_em_targets[robot['id']]
    
    elif robot['state'] == 'moving_to_observe':
        # Moving plant to observation station
        target = robot['target_col']
        if robot['col'] < target:
            robot['col'] = min(robot['col'] + ROBOT_SPEED, target)
        elif robot['col'] > target:
            robot['col'] = max(robot['col'] - ROBOT_SPEED, target)
        
        if robot['col'] == target:
            robot['state'] = 'observing'
            robot['ticks_remaining'] = OBSERVATION_TICKS
    
    elif robot['state'] == 'observing':
        robot['ticks_remaining'] -= 1
        if robot['ticks_remaining'] <= 0:
            # Observation complete - record it
            if robot['holding_plant']:
                plant_id = robot['holding_plant']
                tick = simulation_state['tick']
                
                # Determine effect parameters based on where it was exposed
                effect_type = 'Control'
                effect_props = {}
                
                exposure_info = em_zone_exposure.get(plant_id)
                if exposure_info:
                    # It was in an EM zone
                    effects_grid = get_effects_grid()
                    row, col = exposure_info['row'], exposure_info['col']
                    
                    # Fetch effect details from API/Grid (simulated lookup)
                    with get_h5_file() as f:
                        if 'sequencer_effects_properties' in f:
                            grp = f['sequencer_effects_properties']
                            # Find effect at this location
                            # This is a simplification; ideally we link effect UUIDs to grid
                            # But for now we use the effects_grid content which is effect ID?
                            # Or we can just grab properties from the cached grid if stored
                            pass

                    # Retrieve properties (simulated/random for now if not linked)
                    # Use get_effect_at(row, col) helper if it exists, or just use grid effect ID
                    effect_uuid = effects_grid[row][col]
                    if effect_uuid:
                         # Get type and props
                         try:
                             with get_h5_file() as f:
                                 if 'sequencer_effects_properties' in f and effect_uuid in f['sequencer_effects_properties']:
                                     eff_grp = f['sequencer_effects_properties'][effect_uuid]
                                     effect_type = eff_grp.attrs.get('type', 'Control')
                                     # Load all attrs
                                     for k in eff_grp.attrs:
                                         if k != 'type':
                                             effect_props[k] = eff_grp.attrs[k]
                         except:
                             pass
                
                # Calculate growth
                growth = calculate_plant_growth(plant_id, tick, effect_type, effect_props)
                
                # Record to HDF5
                record_growth_data(plant_id, tick, growth, effect_type, effect_props)
                
                # Add to SSE buffer
                with _growth_data_lock:
                    _growth_data_buffer.append({
                        'plant_id': plant_id,
                        'tick': tick,
                        'growth': growth,
                        'effect_type': effect_type,
                        **effect_props
                    })
                    # Keep buffer size reasonable
                    if len(_growth_data_buffer) > 1000:
                        _growth_data_buffer.pop(0)
                _new_data_event.set()

                simulation_state['plants_observed'].append({
                    'plant_id': plant_id,
                    'tick': tick,
                    'growth': growth
                })
                # Keep only last 100 observations
                if len(simulation_state['plants_observed']) > 100:
                    simulation_state['plants_observed'] = simulation_state['plants_observed'][-100:]
            
            # Move to return location
            robot['state'] = 'moving_to_return'
            # Return to the original storage location if available
            robot['target_col'] = float(robot.get('storage_return_col', robot.get('original_col', 0)))
    
    elif robot['state'] == 'moving_to_return':
        # Moving plant back to storage after observation
        target = robot['target_col']
        if robot['col'] < target:
            robot['col'] = min(robot['col'] + ROBOT_SPEED, target)
        elif robot['col'] > target:
            robot['col'] = max(robot['col'] - ROBOT_SPEED, target)
        
        if robot['col'] == target:
            robot['state'] = 'putting_down'
            robot['ticks_remaining'] = PUTDOWN_TICKS
    
    elif robot['state'] == 'putting_down':
        robot['ticks_remaining'] -= 1
        if robot['ticks_remaining'] <= 0:
            # Put down plant - try to place in storage area
            if robot['holding_plant']:
                grid = get_sequencer_grid()
                col = int(robot['target_col'] + 0.5)
                placed = False
                
                # Try the target column first
                if 0 <= col < 8 and not grid[robot['row']][col]:
                    grid[robot['row']][col] = robot['holding_plant']
                    set_sequencer_grid(grid)
                    placed = True
                else:
                    # Find any empty storage slot
                    for alt_col in STORAGE_COLS:
                        if not grid[robot['row']][alt_col]:
                            grid[robot['row']][alt_col] = robot['holding_plant']
                            set_sequencer_grid(grid)
                            placed = True
                            break
                
                if placed:
                    # Clean up EM exposure tracking
                    if robot['holding_plant'] in em_zone_exposure:
                        del em_zone_exposure[robot['holding_plant']]
                    robot['holding_plant'] = None
            
            robot['state'] = 'idle'
            robot['target_col'] = None
            robot['storage_return_col'] = None
            robot['original_col'] = None

# EM zone exposure tracking
em_zone_exposure = {}  # {plant_id: {'tick_placed': int, 'row': int, 'col': int}}

def init_em_zone_exposure_from_grid():
    """Initialize em_zone_exposure from current grid state on startup."""
    global em_zone_exposure
    grid = get_sequencer_grid()
    for row in range(2):
        for col in EM_EXPOSURE_COLS:
            plant_id = grid[row][col]
            if plant_id and plant_id not in em_zone_exposure:
                em_zone_exposure[plant_id] = {
                    'tick_placed': 0,
                    'row': row,
                    'col': col,
                    'original_storage_col': 0
                }
    if em_zone_exposure:
        import sys
        print(f"[INIT] Found {len(em_zone_exposure)} plants already in EM zone", file=sys.stderr)

init_em_zone_exposure_from_grid()

def assign_observation_task():
    """Try to assign observation tasks to idle robots with better coordination"""
    global robot_em_targets
    grid = get_sequencer_grid()
    effects_grid = get_effects_grid()
    
    for row in range(2):
        # Get all idle robots for this row
        idle_robots = [r for r in simulation_state['robots'] if r['row'] == row and r['state'] == 'idle']
        
        for robot in idle_robots:
            current_tick = simulation_state['tick']
            
            # Build map of plant_id -> last observation tick for cooldown check
            last_obs_tick = {}
            for obs in simulation_state['plants_observed']:
                last_obs_tick[obs['plant_id']] = obs['tick']
            
            # PRIORITY 1: Find plants in EM zone that have been exposed long enough
            for col in ROBOT_EM_PREFERENCES.get(robot['id'], EM_EXPOSURE_COLS):
                plant_id = grid[row][col]
                can_observe = plant_id and (
                    plant_id not in last_obs_tick or 
                    current_tick - last_obs_tick[plant_id] >= OBSERVATION_COOLDOWN_TICKS
                )
                if can_observe:
                    exposure_info = em_zone_exposure.get(plant_id)
                    if exposure_info and (current_tick - exposure_info['tick_placed']) >= EM_EXPOSURE_TICKS:
                        robot['state'] = 'moving_to_pickup'
                        robot['target_col'] = float(col)
                        robot['original_col'] = float(col)
                        robot['storage_return_col'] = exposure_info.get('original_storage_col', col)
                        break
            
            if robot['state'] != 'idle':
                continue
            
            # PRIORITY 2: Move plants from storage to EM zone
            target_em_col = get_preferred_em_col(robot['id'], grid, row)
            
            if target_em_col is not None and effects_grid[row][target_em_col]:
                # Find a plant in storage to move
                for col in STORAGE_COLS:
                    plant_id = grid[row][col]
                    if plant_id and plant_id not in em_zone_exposure:
                        robot['state'] = 'moving_to_pickup'
                        robot['target_col'] = float(col)
                        robot['original_col'] = float(col)
                        robot['em_target_col'] = float(target_em_col)
                        robot['task_type'] = 'em_exposure'
                        # Register target to prevent other robots from targeting same col
                        robot_em_targets[robot['id']] = target_em_col
                        break

# --- Simulation Loop Logic ---

def simulation_tick():
    """Advance simulation by one tick"""
    with simulation_lock:
        simulation_state['tick'] += 1
        
        # Update robots
        for robot in simulation_state['robots']:
            update_robot(robot)
            
        # Assign new tasks
        assign_observation_task()
        
        # Every 25 ticks, update the test queue
        if simulation_state['tick'] % 25 == 0:
            update_test_queue()
            
        # Save state occasionally
        if simulation_state['tick'] % 100 == 0:
            save_simulation_state()

def run_simulation():
    """Main simulation loop running in a thread"""
    import time
    print("Simulation thread started")
    
    while True:
        try:
            should_run = False
            speed = 1.0
            with simulation_lock:
                should_run = simulation_state['running']
                speed = simulation_state['speed']
            
            if should_run:
                start_time = time.time()
                simulation_tick()
                end_time = time.time()
                
                # Calculate sleep time based on speed setting
                # Speed is ticks per second
                if speed > 0:
                    target_interval = 1.0 / speed
                    actual_process_time = end_time - start_time
                    sleep_time = max(0, target_interval - actual_process_time)
                    if sleep_time > 0:
                        time.sleep(sleep_time)
                else:
                    time.sleep(1.0) # Should not happen, but safe fallback
            else:
                time.sleep(0.5) # Idle wait
                
        except Exception as e:
            print(f"Error in simulation loop: {e}")
            time.sleep(1.0)

def start_simulation():
    """Start the simulation thread if needed, and set running=True"""
    global simulation_thread
    with simulation_lock:
        simulation_state['running'] = True
        if simulation_thread is None or not simulation_thread.is_alive():
            simulation_thread = threading.Thread(target=run_simulation, daemon=True)
            simulation_thread.start()
    return True

def stop_simulation():
    """Pause the simulation"""
    with simulation_lock:
        simulation_state['running'] = False
    return True

def reset_simulation():
    """Reset simulation state"""
    with simulation_lock:
        simulation_state['running'] = False
        simulation_state['tick'] = 0
        simulation_state['robots'] = [
            {'id': 0, 'row': 0, 'col': 0.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0},
            {'id': 1, 'row': 0, 'col': 4.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0},
            {'id': 2, 'row': 1, 'col': 0.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0},
            {'id': 3, 'row': 1, 'col': 4.0, 'target_col': None, 'state': 'idle', 'holding_plant': None, 'ticks_remaining': 0}
        ]
        simulation_state['plants_observed'] = []
    
    # Clear SSE buffer
    with _growth_data_lock:
        _growth_data_buffer.clear()
        
    return True

# Auto-start simulation thread on load (paused)
start_simulation()
stop_simulation()


In [ ]:
@app.route('/api/clear-all', methods=['POST'])
def clear_all_data():
    """Clear all sequencer data, effects, growth data, and reset simulation"""
    import sys
    try:
        # Stop simulation first
        with simulation_lock:
            simulation_state['running'] = False

        # Clear sequencer grid (plants)
        grid = get_sequencer_grid()
        for row in range(2):
            for col in range(12):
                grid[row][col] = ""
        set_sequencer_grid(grid)

        # Clear effects grid
        effects_grid = get_effects_grid()
        with get_h5_file() as f:
            effects_props_group = f.require_group("sequencer_effects_properties")
            # Delete all effect groups
            effect_uuids = list(effects_props_group.keys())
            for uuid in effect_uuids:
                del effects_props_group[uuid]

            # Clear effects grid
            for row in range(2):
                for col in range(12):
                    effects_grid[row][col] = ""
            set_effects_grid(effects_grid)

        # Clear all plant growth data
        with get_h5_file() as f:
            if "plant_growth_data" in f:
                del f["plant_growth_data"]
                print("[CLEAR-ALL] Deleted plant_growth_data group", file=sys.stderr)

        # Clear in-memory growth data (restore defaultdict to avoid KeyErrors)
        global plant_growth_data, _recorded_observations
        from collections import defaultdict
        plant_growth_data = defaultdict(lambda: {'ticks': [], 'growth': [], 'em_effect': None, 'em_properties': {}})
        _recorded_observations = set()

        # Reset points system
        global points_system
        points_system = {
            'data_points': {},
            'target_coverage': 10,
            'exploration_bonus': 5
        }

        # Clear test queue
        global test_queue
        test_queue = []

        # Reset em_zone_exposure
        global em_zone_exposure
        em_zone_exposure = {}

        # Reset simulation state
        reset_simulation()

        print("[CLEAR-ALL] All data cleared successfully", file=sys.stderr)

        return jsonify({
            'success': True,
            'message': 'All sequencer data, effects, growth data, and simulation state cleared'
        }), 200
    except Exception as e:
        import traceback
        error_msg = traceback.format_exc()
        print(f"[CLEAR-ALL ERROR] {error_msg}", file=sys.stderr)
        return jsonify({"error": str(e), "traceback": error_msg}), 500

In [ ]:
from flask import Response

@app.route('/api/plant-growth/stream')
def stream_plant_growth():
    """Server-Sent Events endpoint for growth data streaming"""
    def generate():
        # Yield connection message
        yield f"data: {{\"connected\": true}}\n\n"
        
        while True:
            # Wait for new data or timeout
            if _new_data_event.wait(timeout=1.0):
                # Clear event to wait again next time
                _new_data_event.clear()
                
                # Fetch new data points from buffer
                new_points = []
                with _growth_data_lock:
                    if _growth_data_buffer:
                        new_points = list(_growth_data_buffer)
                        _growth_data_buffer.clear()
                
                if new_points:
                    for point in new_points:
                        import json
                        # Convert any numpy types before dumping
                        # But our buffer likely has native types already if correctly populated
                        # Just in case:
                        safe_point = {k: convert_to_native(v) for k, v in point.items()}
                        yield f"data: {json.dumps(safe_point)}\n\n"
            
            # Send heartbeat to keep connection alive
            yield ": heartbeat\n\n"

    return Response(generate(), mimetype='text/event-stream')


In [ ]:
# Helper function to convert numpy types to Python native types
def convert_to_native(value):
    """Convert numpy types to native Python types for JSON serialization"""
    import numpy as np
    if isinstance(value, (np.integer, np.int64, np.int32)):
        return int(value)
    elif isinstance(value, (np.floating, np.float64, np.float32)):
        return float(value)
    elif isinstance(value, np.ndarray):
        return value.tolist()
    elif isinstance(value, bytes):
        return value.decode('utf-8')
    return value

def generate_random_em_effect():
    """Generate a random EM effect type and properties for testing"""
    import random
    effect_types = ['AC', 'DC', 'AMF', 'CMF']
    effect_type = random.choice(effect_types)
    
    if effect_type == 'AC':
        properties = {
            'ac_frequency': round(random.uniform(50, 60), 1),   # Hz
            'ac_voltage': round(random.uniform(100, 240), 1),   # V
            'ac_phase': round(random.uniform(0, 360), 1)        # degrees
        }
    elif effect_type == 'DC':
        properties = {
            'dc_voltage': round(random.uniform(5, 24), 1),      # V
            'dc_current': round(random.uniform(0.1, 2.0), 2)    # A
        }
    elif effect_type == 'AMF':
        properties = {
            'amf_frequency': round(random.uniform(1, 100), 1),   # Hz
            'amf_amplitude': round(random.uniform(0.5, 5.0), 2), # arbitrary units
            'amf_phase': round(random.uniform(0, 360), 1)        # degrees
        }
    elif effect_type == 'CMF':
        properties = {
            'cmf_strength': round(random.uniform(0.1, 2.0), 2),  # Tesla or arbitrary
            'cmf_direction': round(random.uniform(0, 360), 1)    # degrees
        }
    
    return (effect_type, properties)

@app.route('/api/plant-growth/unified', methods=['GET'])
def get_unified_plant_growth():
    """
    Fetch all plant growth data with unified EM parameters.
    Returns data in format suitable for 3D visualization.
    """
    try:
        data_points = []
        
        with get_h5_file() as f:
            if 'plant_growth' not in f:
                return jsonify({"data_points": [], "count": 0})
            
            growth_group = f['plant_growth']
            
            for obs_key in growth_group.keys():
                obs = growth_group[obs_key]
                
                # Extract and convert all attributes to native types
                plant_id = obs.attrs.get('plant_id', '')
                if isinstance(plant_id, bytes):
                    plant_id = plant_id.decode('utf-8')
                
                effect_type = obs.attrs.get('effect_type', 'Control')
                if isinstance(effect_type, bytes):
                    effect_type = effect_type.decode('utf-8')
                
                point = {
                    'plant_id': plant_id,
                    'tick': convert_to_native(obs.attrs.get('tick', 0)),
                    'growth': convert_to_native(obs.attrs.get('growth', 0)),
                    'effect_type': effect_type,
                    # AC parameters
                    'ac_frequency': convert_to_native(obs.attrs.get('ac_frequency', 0)),
                    'ac_voltage': convert_to_native(obs.attrs.get('ac_voltage', 0)),
                    'ac_phase': convert_to_native(obs.attrs.get('ac_phase', 0)),
                    # DC parameters
                    'dc_voltage': convert_to_native(obs.attrs.get('dc_voltage', 0)),
                    'dc_current': convert_to_native(obs.attrs.get('dc_current', 0)),
                    # AMF parameters
                    'amf_frequency': convert_to_native(obs.attrs.get('amf_frequency', 0)),
                    'amf_amplitude': convert_to_native(obs.attrs.get('amf_amplitude', 0)),
                    'amf_phase': convert_to_native(obs.attrs.get('amf_phase', 0)),
                    # CMF parameters
                    'cmf_strength': convert_to_native(obs.attrs.get('cmf_strength', 0)),
                    'cmf_direction': convert_to_native(obs.attrs.get('cmf_direction', 0))
                }
                
                data_points.append(point)
        
        return jsonify({
            "data_points": data_points,
            "count": len(data_points)
        })
    
    except Exception as e:
        import traceback
        return jsonify({"error": str(e), "traceback": traceback.format_exc()}), 500

In [ ]:
# Simulation API endpoints

@app.route('/api/simulation/state', methods=['GET'])
def get_simulation_state():
    """Get current simulation state"""
    try:
        with simulation_lock:
            return jsonify({
                'running': simulation_state['running'],
                'tick': simulation_state['tick'],
                'speed': simulation_state['speed'],
                'robots': simulation_state['robots'],
                'plants_observed': simulation_state['plants_observed'][-10:],  # Last 10
                'tile_allocation': get_tile_allocation()
            }), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/simulation/start', methods=['POST'])
def start_simulation_endpoint():
    """Start the simulation"""
    try:
        started = start_simulation()
        return jsonify({'success': started, 'running': simulation_state['running']}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/simulation/pause', methods=['POST'])
def pause_simulation_endpoint():
    """Pause the simulation"""
    try:
        stopped = stop_simulation()
        return jsonify({'success': stopped, 'running': simulation_state['running']}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/simulation/reset', methods=['POST'])
def reset_simulation_endpoint():
    """Reset the simulation"""
    try:
        reset = reset_simulation()
        return jsonify({'success': reset, 'tick': 0}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/simulation/speed', methods=['PUT'])
def set_simulation_speed():
    """Set simulation speed (ticks per second)"""
    try:
        data = request.get_json()
        speed = data.get('speed', 1.0)
        
        # Clamp speed between 0.1 and 1000.0 (ultra-fast for debugging)
        speed = max(0.1, min(1000.0, float(speed)))
        
        with simulation_lock:
            simulation_state['speed'] = speed
        
        return jsonify({'speed': speed}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/simulation/tick', methods=['POST'])
def manual_tick():
    """Manually advance one tick (for debugging/testing)"""
    try:
        with simulation_lock:
            was_running = simulation_state['running']
            simulation_state['running'] = True
        
        simulation_tick()
        
        with simulation_lock:
            simulation_state['running'] = was_running
            return jsonify({
                'tick': simulation_state['tick'],
                'robots': simulation_state['robots']
            }), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/api/tile-allocation', methods=['GET'])
def get_tile_allocation_endpoint():
    """Get tile allocation info"""
    try:
        return jsonify(get_tile_allocation()), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500


In [ ]:
if __name__ == "__main__":
    print(f"Starting Flask server on http://localhost:5000")
    print(f"API endpoints:")
    print(f"  POST /api/plants - Create a new plant")
    print(f"  GET /api/plants - List all plants")
    print(f"  GET /api/sequencer - Get sequencer grid")
    print(f"  PUT /api/sequencer - Update sequencer position")
    print(f"  GET /api/effects - Get effects grid")
    print(f"  PUT /api/effects - Update effects grid position")
    print(f"  GET /api/effects/<row>/<col>/properties - Get effect properties")
    print(f"  PUT /api/effects/<row>/<col>/properties - Update effect properties")
    print(f"  GET /api/simulation/state - Get simulation state")
    print(f"  POST /api/simulation/start - Start simulation")
    print(f"  POST /api/simulation/pause - Pause simulation")
    print(f"  POST /api/simulation/reset - Reset simulation")
    print(f"  PUT /api/simulation/speed - Set simulation speed")
    print(f"  POST /api/simulation/tick - Manual tick")
    print(f"  GET /api/tile-allocation - Get tile allocation")
    app.run(host='0.0.0.0', port=5000, debug=True)
    print(f"  POST /api/clear-all - Clear all sequencer data, effects, growth data, and reset simulation")
